<a href="https://colab.research.google.com/github/thuviettran/demo-github/blob/main/movie_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# Hybrid Movie Recommender tối ưu cho MovieLens 1M
# Bao gồm: re-index ID, chuẩn hóa rating, genre embedding, early stopping

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import (
    Input, Embedding, Flatten, Dense, Concatenate,
    Dropout, GlobalAveragePooling1D
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping

# =====================
# 1. Load dữ liệu .dat
# =====================
ratings = pd.read_csv(
    "ratings.dat",
    sep="::",
    engine="python",
    names=["userId", "movieId", "rating", "timestamp"]
)

movies = pd.read_csv(
    "movies.dat",
    sep="::",
    engine="python",
    names=["movieId", "title", "genres"],
    encoding="latin-1"
)

# =====================
# 2. Re-index ID (quan trọng)
# Fit encoder trên TẬP HỢP movieId của cả ratings + movies để tránh unseen labels
# =====================
user_encoder = LabelEncoder()
movie_encoder = LabelEncoder()

ratings["userId"] = user_encoder.fit_transform(ratings["userId"])

# Fit movie encoder trên union movieId
all_movie_ids = pd.concat([ratings["movieId"], movies["movieId"]]).unique()
movie_encoder.fit(all_movie_ids)

ratings["movieId"] = movie_encoder.transform(ratings["movieId"])
movies["movieId"] = movie_encoder.transform(movies["movieId"])

num_users = ratings["userId"].nunique()
num_movies = len(all_movie_ids)

# =====================
# 3. Xử lý genre → danh sách index
# =====================
movies["genre_list"] = movies["genres"].apply(lambda x: x.split("|"))

all_genres = sorted({g for sub in movies["genre_list"] for g in sub})
genre_to_idx = {g: i + 1 for i, g in enumerate(all_genres)}  # 0 = padding

max_genres = movies["genre_list"].apply(len).max()
num_genres = len(all_genres) + 1


def encode_genres(genre_list):
    idxs = [genre_to_idx[g] for g in genre_list]
    return idxs + [0] * (max_genres - len(idxs))


movies["genre_encoded"] = movies["genre_list"].apply(encode_genres)

# =====================
# 4. Merge dữ liệu
# =====================
df = ratings.merge(movies[["movieId", "genre_encoded"]], on="movieId")

# =====================

# Sửa lại phần 5 & 6
y = df["rating"].values.astype("float32")
user_input = df["userId"].values
movie_input = df["movieId"].values
genre_input = np.stack(df["genre_encoded"].values)

# Chia Train/Test TRƯỚC
(
    user_train, user_test,
    movie_train, movie_test,
    genre_train, genre_test,
    y_train_raw, y_test_raw
) = train_test_split(
    user_input, movie_input, genre_input, y,
    test_size=0.2,
    random_state=42
)

# Chuẩn hóa SAU KHI chia (chỉ dùng tham số của tập Train)
y_mean = y_train_raw.mean()
y_std = y_train_raw.std()

y_train = (y_train_raw - y_mean) / y_std
y_test = (y_test_raw - y_mean) / y_std # Chuẩn hóa tập test bằng mean/std của tập train
# =====================
# 7. Xây dựng Hybrid Model tối ưu
# =====================
embedding_dim = 64

user_in = Input(shape=(1,))
movie_in = Input(shape=(1,))
genre_in = Input(shape=(max_genres,))

# User & Movie embedding (Collaborative Filtering)
user_emb = Embedding(num_users, embedding_dim)(user_in)
movie_emb = Embedding(num_movies, embedding_dim)(movie_in)

user_vec = Flatten()(user_emb)
movie_vec = Flatten()(movie_emb)

# Genre embedding (Content-based)
genre_emb = Embedding(num_genres, 32, mask_zero=True)(genre_in)
genre_vec = GlobalAveragePooling1D()(genre_emb)

# Combine tất cả
x = Concatenate()([user_vec, movie_vec, genre_vec])

# Deep layers
x = Dense(300, activation="relu")(x)
x = Dropout(0.5)(x)
x = Dense(200, activation="relu")(x)
x = Dense(64, activation="relu")(x)

out = Dense(1, activation="linear")(x)

model = Model(inputs=[user_in, movie_in, genre_in], outputs=out)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="mse",
    metrics=["mae"]
)

model.summary()

# =====================
# 8. Train với EarlyStopping
# =====================
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    [user_train, movie_train, genre_train],
    y_train,
    validation_split=0.1,
    epochs=20,
    batch_size=512,
    callbacks=[early_stop],
    verbose=1
)

# =====================
# 9. Evaluate
# =====================
loss, mae = model.evaluate(
    [user_test, movie_test, genre_test],
    y_test,
    verbose=0
)

rmse = np.sqrt(loss)

print(f"Test RMSE: {rmse:.4f}")
print(f"Test MAE: {mae:.4f}")

# =====================
# =====================
# Lấy giá trị tệp test và so sánh dự đoán
# =====================

# 1. Yêu cầu mô hình dự đoán trên tập test
y_pred_normalized = model.predict([user_test, movie_test, genre_test], verbose=0)

# 2. Scale ngược giá trị dự đoán về lại thang điểm 1-5 sao
y_pred_real = (y_pred_normalized.flatten() * y_std) + y_mean

# 3. Lấy giá trị thực tế (chính là y_test_raw ở bước chia dữ liệu)
y_true_real = y_test_raw

# 4. Đưa tất cả vào một DataFrame để dễ quan sát
test_results_df = pd.DataFrame({
    'User_ID_Encoded': user_test,
    'Movie_ID_Encoded': movie_test,
    'True_Rating': y_true_real,
    'Predicted_Rating': np.round(y_pred_real, 2), # Làm tròn 2 chữ số thập phân
    'Sai_Số': np.round(abs(y_true_real - y_pred_real), 2) # Tính độ lệch tuyệt đối
})

# Hiển thị 10 dòng đầu tiên của tập test
print("--- 10 GIAO DỊCH TRONG TẬP TEST ---")
print(test_results_df.head(200))
# 10. Hàm gợi ý phim
# =====================
movies_full = movies.copy()


def recommend(user_id_raw, top_k=10):
    user_id = user_encoder.transform([user_id_raw])[0]

    user_arr = np.full(len(movies_full), user_id)
    movie_arr = movies_full["movieId"].values
    genre_arr = np.stack(movies_full["genre_encoded"].values)

    preds = model.predict([user_arr, movie_arr, genre_arr], verbose=0).flatten()

    # scale ngược rating
    preds = preds * y_std + y_mean

    movies_full["pred_rating"] = preds

    return movies_full.sort_values("pred_rating", ascending=False).head(top_k)[
        ["title", "genres", "pred_rating"]
    ]


# Ví dụ gợi ý cho user gốc = 1
print("--- Gợi ý phim cho user gốc = 1 ---")
print(recommend(1))


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_12      │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_13      │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_14      │ (None, 6)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_12        │ (None, 1, 64)     │    386,560 │ input_layer_12[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_13        │ (None, 1, 64)     │    248,512 │ input_layer_13[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_14        │ (None, 6, 32)     │        608 │ input_layer_14[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_4         │ (None, 6)         │          0 │ input_layer_14[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_8 (Flatten) │ (None, 64)        │          0 │ embedding_12[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_9 (Flatten) │ (None, 64)        │          0 │ embedding_13[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 32)        │          0 │ embedding_14[0][… │
│ (GlobalAveragePool… │                   │            │ not_equal_4[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_4       │ (None, 160)       │          0 │ flatten_8[0][0],  │
│ (Concatenate)       │                   │            │ flatten_9[0][0],  │
│                     │                   │            │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_16 (Dense)    │ (None, 300)       │     48,300 │ concatenate_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 300)       │          0 │ dense_16[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_17 (Dense)    │ (None, 200)       │     60,200 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_18 (Dense)    │ (None, 64)        │     12,864 │ dense_17[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_19 (Dense)    │ (None, 1)         │         65 │ dense_18[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 757,109 (2.89 MB)

 Trainable params: 757,109 (2.89 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - loss: 0.6921 - mae: 0.6579 - val_loss: 0.6522 - val_mae: 0.6370
Epoch 2/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.6370 - mae: 0.6291 - val_loss: 0.6304 - val_mae: 0.6291
Epoch 3/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.6089 - mae: 0.6154 - val_loss: 0.6149 - val_mae: 0.6211
Epoch 4/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.5880 - mae: 0.6047 - val_loss: 0.6079 - val_mae: 0.6176
Epoch 5/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.5706 - mae: 0.5956 - val_loss: 0.6021 - val_mae: 0.6131
Epoch 6/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 0.5550 - mae: 0.5872 - val_loss: 0.5961 - val_mae: 0.6061
Epoch 7/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.5408 - mae: 0.5797 - val_loss: 0.5963 - val_mae: 0.6089
Epoch 8/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.5282 - mae: 0.5728 - val_loss: 0.5966 - val_mae: 0.6093
Epoch 9/20
1407/1407 ━━━━━━━━━━━━━━━━━━

In [5]:
print("--- Gợi ý phim cho user gốc = 1 ---")
print(recommend(1))


--- Gợi ý phim cho user gốc = 1 ---
                                                title  \
315                  Shawshank Redemption, The (1994)   
2693                          Sixth Sense, The (1999)   
523                           Schindler's List (1993)   
3538                         One Little Indian (1973)   
108                                 Braveheart (1995)   
352                               Forrest Gump (1994)   
1180                   Raiders of the Lost Ark (1981)   
257         Star Wars: Episode IV - A New Hope (1977)   
2087  Best Man, The (Il Testimone dello sposo) (1997)   
2836                                   Sanjuro (1962)   

                               genres  pred_rating  
315                             Drama     4.678160  
2693                         Thriller     4.587743  
523                         Drama|War     4.567666  
3538             Comedy|Drama|Western     4.557380  
108                  Action|Drama|War     4.546721  
352               

In [ ]:
import pickle

# Lưu mô hình Deep Learning
model.save("movie_recommender.keras")

# Lưu các bộ mã hóa (Encoder) và dữ liệu cần thiết để phục vụ Web
artifacts = {
    "user_encoder": user_encoder,
    "movies_full": movies, # Lưu lại thông tin phim
    "y_mean": y_mean,
    "y_std": y_std
}

with open("artifacts.pkl", "wb") as f:
    pickle.dump(artifacts, f)

print("Đã lưu model (movie_recommender.h5) và artifacts (artifacts.pkl) thành công!")

In [ ]:
print(test_results_df.head(4000))

In [ ]:
# Xuất ra file Excel
test_results_df.to_excel("Ket_Qua_Tap_Test.xlsx", index=False)

print("Đã xuất thành công ra file Ket_Qua_Tap_Test.xlsx!")

In [3]:
# =====================
# In ra bộ phim có nhiều thể loại nhất
# =====================

# 1. Tạo một cột mới đếm số lượng thể loại của từng phim
movies['genre_count'] = movies['genre_list'].apply(len)

# 2. Tìm ra (các) bộ phim có số lượng thể loại bằng với max_genres
movies_with_most_genres = movies[movies['genre_count'] == max_genres]

# 3. In kết quả ra màn hình (chỉ lấy các cột cần thiết cho dễ nhìn)
print(f"--- CÁC BỘ PHIM CÓ NHIỀU THỂ LOẠI NHẤT (Số lượng: {max_genres}) ---")
print(movies_with_most_genres[['title', 'genres', 'genre_count']])

--- CÁC BỘ PHIM CÓ NHIỀU THỂ LOẠI NHẤT (Số lượng: 6) ---
                                    title  \
1187  Transformers: The Movie, The (1986)   

                                               genres  genre_count  
1187  Action|Animation|Children's|Sci-Fi|Thriller|War            6  
